# Using the saved models and checking the results

This notebook loads saved E2M models, predicts with them, and compares the predictions and metrics with the saved results. Run it from the root of the analysis code, after a run (`python run_all.py`) or after unzipping the Zenodo release into this folder (`tcga_results.zip`, `tcga_models.zip`, `tcga_cv_fold_models.zip`, `external.zip`).

The TCGA data are downloaded into `Runs/TCGA/data` on first use.

In [1]:
from pathlib import Path

import pandas as pd

import e2m
from e2m.config import load_config
from e2m.evaluation import evaluate_predictions

ROOT = Path.cwd()
TCGA = ROOT / "Runs" / "TCGA"
TCGA_CONFIG = TCGA / "config" / "e2m.yaml"  # the manuscript's TCGA settings (CPM, log1p, protein-coding genes)
EXTERNAL = ROOT / "Runs" / "External" / "output"

## 1. Predict with a saved model

A model folder holds the network weights with the training-set scaling of each gene (`model.pt`), the gene and target order (`features.json`, `targets.json`) and the full configuration (`model_metadata.json`). `Runs/TCGA/output/<cancer>/model` is trained on all samples of that cancer type (`all` = pan-cancer).

In [2]:
model = e2m.E2MModel.load(TCGA / "output" / "BRCA" / "model")
print(len(model.features), "genes,", len(model.targets), "targets:", ", ".join(model.targets))

data = e2m.Dataset.from_tcga(["BRCA"], data_dir=TCGA / "data", config=TCGA_CONFIG, with_tmb=False)
scores = model.predict(data)        # samples x targets, scores between 0 and 1
embeddings = model.embed(data)      # samples x 256, the shared representation
print(scores.shape, embeddings.shape)
scores.head()

19938 genes, 16 targets: PIK3CA, TP53, TTN, CDH1, GATA3, MUC16, KMT2C, MAP3K1, SYNE1, PTEN, RYR2, FLG, HMCN1, SPTA1, USH2A, DMD


(788, 16) (788, 256)


,PIK3CA,TP53,TTN,CDH1,GATA3,MUC16,KMT2C,MAP3K1,SYNE1,PTEN,RYR2,FLG,HMCN1,SPTA1,USH2A,DMD
sample_id,,,,,,,,,,,,,,,,
TCGA-D8-A146-01A,0.834173,0.079642,0.243767,0.582082,0.334055,0.168634,0.485102,0.561848,0.289886,0.350904,0.296259,0.121629,0.148951,0.212092,0.202772,0.319213
TCGA-AQ-A0Y5-01A,0.523738,0.324900,0.307884,0.138324,0.347302,0.189253,0.265996,0.191020,0.278036,0.282348,0.245817,0.360177,0.162455,0.154402,0.172883,0.231561
TCGA-C8-A274-01A,0.640774,0.477617,0.474755,0.562768,0.559547,0.620877,0.765871,0.504960,0.690669,0.376608,0.467731,0.415226,0.385890,0.759679,0.291494,0.588232
TCGA-AC-A62V-01A,0.279696,0.189045,0.472010,0.222333,0.702291,0.371123,0.280829,0.229184,0.297862,0.275256,0.199534,0.313539,0.386348,0.174110,0.169605,0.245198
TCGA-A2-A0YM-01A,0.120185,0.947662,0.563853,0.083027,0.073364,0.446146,0.474159,0.243508,0.596852,0.371433,0.589364,0.766152,0.553774,0.781134,0.565265,0.647809


## 2. Check the TCGA cross-validation

Each sample's saved out-of-fold prediction (`cv/oof_probabilities.csv`) was made by the model of the fold that held it out (`cv/folds/fold_<k>`, with the folds in `cv/fold_assignments.csv`). Predicting again with those models and recomputing the metrics should give the saved values.

In [3]:
def compare(predicted, saved, labels, saved_metrics):
    """Largest difference from the saved predictions, and recomputed vs saved normalized AUPRC per target."""
    saved = saved.loc[predicted.index, predicted.columns]
    labels = labels.loc[predicted.index, predicted.columns]
    metrics = evaluate_predictions(labels, (predicted >= 0.5).astype(int), predicted, list(predicted.columns))
    print(f"{predicted.shape[0]} samples x {predicted.shape[1]} targets; "
          f"largest difference from the saved predictions: {(predicted - saved).abs().max().max():.1e}")
    auprc = pd.DataFrame({"saved": saved_metrics["normalized_auprc"], "recomputed": metrics["normalized_auprc"]}).dropna()
    print(f"normalized AUPRC, largest difference over {len(auprc)} targets: {(auprc['saved'] - auprc['recomputed']).abs().max():.1e}")
    return auprc.sort_values("saved", ascending=False)


def check_tcga(cohort):
    cv = TCGA / "output" / cohort / "cv"
    cancers = load_config(TCGA_CONFIG)["run"]["cohorts"] if cohort == "all" else [cohort]
    data = e2m.Dataset.from_tcga(cancers, data_dir=TCGA / "data", config=TCGA_CONFIG, with_tmb=False)
    folds = pd.read_csv(cv / "fold_assignments.csv", index_col="sample_id")["fold"]
    predicted = pd.concat(
        e2m.E2MModel.load(cv / "folds" / f"fold_{fold}").predict(data.subset(samples=folds.index[folds == fold]))
        for fold in sorted(folds.unique()))
    return compare(predicted, pd.read_csv(cv / "oof_probabilities.csv", index_col=0), data.mutations,
                   pd.read_csv(cv / "metrics.csv", index_col="target"))


check_tcga("BRCA").head(10)

788 samples x 16 targets; largest difference from the saved predictions: 3.0e-08
normalized AUPRC, largest difference over 16 targets: 9.4e-17


,saved,recomputed
target,,
TP53,0.713652,0.713652
PIK3CA,0.425271,0.425271
CDH1,0.421605,0.421605
GATA3,0.326755,0.326755
MAP3K1,0.262728,0.262728
TTN,0.098007,0.098007
SYNE1,0.090559,0.090559
PTEN,0.042261,0.042261
DMD,0.037982,0.037982


## 3. Check an external cohort

Each external run (`Runs/External/output/<cohort>/<method>`) saves the TCGA model applied to the cohort (`model/`) and that model's input, the cohort's expression after integration (`expression_external.csv.gz`; within-sample gene ranks for `sample_rank`).

In [4]:
def check_external(cohort, method="sample_rank"):
    run = EXTERNAL / cohort / method
    model = e2m.E2MModel.load(run / "model")
    predicted = model.predict(pd.read_csv(run / "expression_external.csv.gz", index_col=0))
    return compare(predicted, pd.read_csv(run / "probabilities.csv.gz", index_col=0),
                   pd.read_csv(run / "labels.csv.gz", index_col=0), pd.read_csv(run / "metrics.csv", index_col="target"))


check_external("HUGO").head(10)

28 samples x 20 targets; largest difference from the saved predictions: 3.0e-08
normalized AUPRC, largest difference over 20 targets: 8.3e-17


,saved,recomputed
target,,
TTN,0.707719,0.707719
ANK3,0.672280,0.672280
FAT3,0.659301,0.659301
DMBT1,0.620142,0.620142
RP1,0.523438,0.523438
PKHD1L1,0.505489,0.505489
CSMD3,0.504335,0.504335
TP53,0.486413,0.486413
COL5A1,0.485648,0.485648


The other runs are checked the same way, e.g. `check_tcga("LUAD")`, `check_tcga("all")` (pan-cancer), `check_external("METABRIC", "combat")` or `check_external("MORRISON", "rank")`; the runs are listed in `Runs/External/output/runs_summary.csv`.

In [5]:
pd.read_csv(EXTERNAL / "runs_summary.csv")[["cohort", "method", "main", "samples", "targets", "median_normalized_auprc"]]

,cohort,method,main,samples,targets,median_normalized_auprc
0,HUGO,sample_rank,True,28,20,0.442547
1,LIU,sample_rank,True,121,20,0.284215
2,RIAZ,sample_rank,True,100,20,0.153459
3,VAN_ALLEN,sample_rank,True,42,20,0.141509
4,MORRISON,sample_rank,False,442,20,0.238880
5,MORRISON,rank,True,442,20,0.290976
6,HUGO,sample_rank_shared,False,28,20,0.458373
7,LIU,sample_rank_shared,False,121,20,0.290460
8,RIAZ,sample_rank_shared,False,100,20,0.275887
9,CPTAC_CMI,sample_rank,True,1988,534,0.197069


## 4. Predict on your own expression table

`predict` also takes a samples x genes DataFrame (gene symbols as columns), processed like the model's training data: for the TCGA models, protein-coding genes as counts per million followed by log1p. Genes are matched to the model's genes by symbol; genes the model does not know are ignored, and missing genes are set to their training mean (at least 80% of the model's genes must be present). Here, TCGA-BRCA expression with 5% of the genes removed:

In [6]:
table = data.expression.drop(columns=data.expression.columns[::20])   # samples x genes, CPM + log1p
partial = model.predict(table)
print(f"{table.shape[1]} of {len(model.features)} genes; largest change in the scores: {(partial - scores).abs().max().max():.3f}")

18941 of 19938 genes; largest change in the scores: 0.025


## 5. Rebuild and rerun

From this folder:

```
python Figures/run_figures.py tcga external singlecell tables   # figures and tables from the saved outputs, no retraining
python run_all.py                                               # everything from the data (see README.md for the files to place by hand)
python run_all.py tcga                                          # one part
```

Side analyses: `python Runs/TCGA/src/shap_stability.py` (SHAP stability, Supplementary Table 2c), `python Runs/External/src/braf_xgboost.py` (BRAF in melanoma) and `python Runs/Benchmark/timing.py [--device cpu]` (run time).